# Day 41: AI That Computes Instead of Guesses

LLMs predict the *next word* — they don't actually calculate. Ask one to count letters or crunch big numbers and it confidently guesses... often wrong.

Today we give Gemini a **code execution** tool. Now it writes Python, *runs* it, and answers with the real computed result.

## Install

In [ ]:
%pip install google-genai --quiet

## Setup

In [17]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def show(response):
    """Print only what matters: the code Gemini wrote, the result of running it,
    and its final answer — skipping the model's verbose 'thinking' narration."""
    code = result = final = None
    for part in response.candidates[0].content.parts:
        if part.executable_code:
            code = part.executable_code.code
        if part.code_execution_result:
            result = part.code_execution_result.output.strip()
        if part.text and part.text.strip():
            final = part.text.strip()   # keep the LAST text part = the real answer
    if code:
        print("🐍 CODE GEMINI WROTE\n" + code)
    if result is not None:
        print("\n▶️  RESULT FROM RUNNING IT\n" + result)
    if final:
        print("\n💬 ANSWER\n" + final)

print("✅ Client ready")

✅ Client ready


## The Problem — It Guesses

A simple task: count the letter *r*. The model has no calculator, so it just... predicts a number.

In [18]:
sentence = "Strawberries and raspberries are red rare berries carried across the river."

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=f"How many 'r' letters are in this sentence? Give just the number.\n\n{sentence}"
)
print("Gemini guesses:", response.text.strip())

Gemini guesses: 11


## The Ground Truth

Let's actually count it ourselves.

In [19]:
print("Real answer:", sentence.lower().count('r'))

Real answer: 17


## The Fix — Code Execution

Add one tool: `code_execution`. Now Gemini writes Python, runs it on Google's servers, and reports the real result.

In [20]:
response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=f"How many 'r' letters are in this sentence? Write and run Python to count exactly.\n\n{sentence}",
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)]
    )
)
show(response)

🐍 CODE GEMINI WROTE
sentence = "Strawberries and raspberries are red rare berries carried across the river."
r_count = sentence.count('r')
print(f'The number of "r" letters in the sentence is: {r_count}')

▶️  RESULT FROM RUNNING IT
The number of "r" letters in the sentence is: 17

💬 ANSWER
The number of 'r' letters in the sentence "Strawberries and raspberries are red rare berries carried across the river." is 17.


## It Scales to Real Math

Not just letter-counting. Ask something no one could eyeball — it computes the exact answer.

In [21]:
response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="What is the sum of all prime numbers below 1000? Write and run Python to compute it.",
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)]
    )
)
show(response)

🐍 CODE GEMINI WROTE
import sympy

# Generate prime numbers up to 999 (below 1000)
primes_below_1000 = list(sympy.primerange(0, 1000))

# Calculate the sum of these prime numbers
total_sum = sum(primes_below_1000)

print(f'{total_sum=}')


▶️  RESULT FROM RUNNING IT
total_sum=76127

💬 ANSWER
**Result:**

The sum of all prime numbers below 1000 is 76127.


## Wrap It in a Function

One helper — ask anything, get a computed (not guessed) answer, and see the code.

In [22]:
def compute(question: str):
    """Ask Gemini to solve it by writing and running code."""
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=question + " Write and run Python to get the exact answer.",
        config=types.GenerateContentConfig(
            tools=[types.Tool(code_execution=types.ToolCodeExecution)]
        )
    )
    for part in response.candidates[0].content.parts:
        if part.code_execution_result:
            return part.code_execution_result.output.strip()
    return response.text

print(compute("What is 47! (47 factorial)?"))

The factorial of 47 is: 258623241511168180642964355153611979969197632389120000000000


## Key Takeaways

1. **LLMs guess, they don't calculate** — they predict tokens, so exact math and counting trip them up
2. **`types.Tool(code_execution=types.ToolCodeExecution)`** — one tool lets Gemini write and run Python for a verified answer
3. **Read the parts** — `executable_code` is what it wrote, `code_execution_result` is what running it produced

---

Next: Day 42 — Coming soon